# Proyecto Final — Sistemas Inteligentes II
## Aprendizaje por Transferencia en Visión por Computador

**Asignatura:** Sistemas Inteligentes II — Universidad de Caldas
**Profesor:** Jorge Alberto Jaramillo Garzón
**Integrantes:** Daniela Agudelo Martínez, Andersson Julián López Martínez

---

### Dominio 
Clasificación de **enfermedades en hojas de tomate** a partir de imágenes, usando un modelo
convolucional preentrenado en ImageNet (**MobileNetV2**) como base de transferencia.

### Pregunta experimental / Hipótesis
> **¿El descongelamiento parcial (fine-tuning) de las últimas capas de MobileNetV2 mejora el
> F1-score frente a usar el modelo únicamente como extractor de características (capas congeladas),
> y cuál es el costo computacional adicional de esa mejora?**

**Hipótesis (H1):** El fine-tuning parcial obtendrá un F1-score *macro* superior al del extractor
de características congelado, a costa de un mayor tiempo de entrenamiento y mayor riesgo de
sobreajuste.

### Configuraciones comparadas
| | Config A — *Feature Extractor* | Config B — *Fine-Tuning parcial* |
|---|---|---|
| Backbone MobileNetV2 | **Congelado** (no entrenable) | Últimas capas **descongeladas** |
| Capas entrenadas | Solo el clasificador (head) | Head + bloque superior del backbone |
| Learning rate | 1e-3 | 1e-3 (head) → 1e-5 (fine-tuning) |

Ambas configuraciones comparten dataset, particiones, semilla, *data augmentation* y arquitectura
del clasificador; **lo único que cambia es la estrategia de transferencia**, lo que aísla la
variable bajo estudio.

## 1. Configuración del entorno e importación de librerías

Este notebook está pensado para ejecutarse tanto en **Google Colab (GPU gratuita)** como en una
**GPU local estándar** (p. ej. RTX 2060). Usamos `tensorflow_datasets` para la **descarga
automática** del dataset PlantVillage, de modo que el experimento sea totalmente reproducible.

In [ ]:
# Instalación de dependencias.
# En Google Colab, TensorFlow ya viene instalado; descomenta solo lo que falte.
#!pip install -q tensorflow tensorflow-datasets scikit-learn matplotlib seaborn pandas

In [1]:
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install numpy pandas matplotlib seaborn scikit-learn tensorflow tensorflow-datasets

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("Todo OK")

TensorFlow: 2.21.0
Todo OK


In [3]:
import sys

print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\ajlop\AppData\Local\Programs\Python\Python311\python.exe


In [ ]:
!{sys.executable} -m pip list

In [ ]:
!where python
!where pip

In [4]:
import os
import time
import random
import platform
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models, Model, Input
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

print("TensorFlow version:", tf.__version__)
print("TFDS version:", tfds.__version__)

TensorFlow version: 2.21.0
TFDS version: 4.9.10


## 2. Reproducibilidad y registro de metadatos

El criterio de evaluación exige reportar como mínimo: modelo, dataset, particiones, tamaño de
entrada, épocas, optimizador y *learning rate*, *batch size*, técnica de transferencia, hardware,
tiempo de entrenamiento y **semilla aleatoria**. Aquí fijamos la semilla y detectamos el hardware;
el resto de metadatos se irá registrando en el diccionario `META` a lo largo del notebook.

In [5]:
# ----------------------------------------------------------------------------
# Hiperparámetros y configuración global del experimento (un solo lugar a tocar)
# ----------------------------------------------------------------------------
SEED          = 42            # Semilla aleatoria para reproducibilidad
IMG_SIZE      = (160, 160)    # Tamaño de entrada esperado por el backbone
BATCH_SIZE    = 32            # Tamaño de lote
MAX_PER_CLASS = 300           # Imágenes por clase (submuestreo para correr rápido en Colab)
VAL_SPLIT     = 0.15          # Proporción de validación
TEST_SPLIT    = 0.15          # Proporción de prueba
EPOCHS_HEAD   = 12            # Épocas para entrenar el clasificador (head)
EPOCHS_FT     = 8             # Épocas adicionales de fine-tuning (solo Config B)
LR_HEAD       = 1e-3          # Learning rate del clasificador
LR_FINETUNE   = 1e-5          # Learning rate (bajo) para el descongelamiento
FINE_TUNE_AT  = 120           # A partir de qué capa de MobileNetV2 se descongela (Config B)


def set_seeds(seed: int = SEED):
    """Fija todas las fuentes de aleatoriedad para que el experimento sea reproducible."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seeds(SEED)
print(f"Semilla aleatoria fijada en: {SEED}")

Semilla aleatoria fijada en: 42


In [6]:
# ----------------------------------------------------------------------------
# Detección de hardware (CPU / GPU) y registro de metadatos del entorno
# ----------------------------------------------------------------------------
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    # Evita que TF reserve toda la VRAM de golpe (útil en GPU local como la RTX 2060)
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(e)
    try:
        gpu_name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", "GPU")
    except Exception:
        gpu_name = "GPU"
    device_str = f"CUDA / GPU ({gpu_name})"
else:
    device_str = "CPU"

META = {
    "modelo_base": "MobileNetV2 (ImageNet)",
    "dataset": "PlantVillage - subconjunto Tomate (tensorflow_datasets)",
    "tamano_entrada": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "semilla": SEED,
    "hardware": device_str,
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
}

print("Hardware detectado:", device_str)
print("Núm. de GPUs disponibles:", len(gpus))
for k, v in META.items():
    print(f"  - {k}: {v}")

Hardware detectado: CPU
Núm. de GPUs disponibles: 0
  - modelo_base: MobileNetV2 (ImageNet)
  - dataset: PlantVillage - subconjunto Tomate (tensorflow_datasets)
  - tamano_entrada: (160, 160)
  - batch_size: 32
  - semilla: 42
  - hardware: CPU
  - python: 3.11.9
  - tensorflow: 2.21.0


In [7]:
import tensorflow as tf
import tensorflow_datasets as tfds

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)

TensorFlow: 2.21.0
TFDS: 4.9.10


In [8]:
import tensorflow_datasets as tfds

print(tfds.list_builders())

['abstract_reasoning', 'accentdb', 'aeslc', 'aflw2k3d', 'ag_news_subset', 'ai2_arc', 'ai2_arc_with_ir', 'ai2dcaption', 'aloha_mobile', 'amazon_us_reviews', 'anli', 'answer_equivalence', 'arc', 'asimov_dilemmas_auto_val', 'asimov_dilemmas_scifi_train', 'asimov_dilemmas_scifi_val', 'asimov_injury_val', 'asimov_multimodal_auto_val', 'asimov_multimodal_manual_val', 'asimov_v2_constraints_with_rationale', 'asimov_v2_constraints_without_rationale', 'asimov_v2_injuries', 'asimov_v2_videos', 'asqa', 'asset', 'assin2', 'asu_table_top_converted_externally_to_rlds', 'austin_buds_dataset_converted_externally_to_rlds', 'austin_sailor_dataset_converted_externally_to_rlds', 'austin_sirius_dataset_converted_externally_to_rlds', 'bair_robot_pushing_small', 'bc_z', 'bccd', 'beans', 'bee_dataset', 'beir', 'berkeley_autolab_ur5', 'berkeley_cable_routing', 'berkeley_fanuc_manipulation', 'berkeley_gnm_cory_hall', 'berkeley_gnm_recon', 'berkeley_gnm_sac_son', 'berkeley_mvp_converted_externally_to_rlds', 'ber

In [9]:
builder = tfds.builder("plant_village")
print(builder.info)

tfds.core.DatasetInfo(
    name='plant_village',
    full_name='plant_village/1.0.2',
    description="""
    The PlantVillage dataset consists of 54303 healthy and unhealthy leaf images
    divided into 38 categories by species and disease.
    
    NOTE: The original dataset is not available from the original source
    (plantvillage.org), therefore we get the unaugmented dataset from a paper that
    used that dataset and republished it. Moreover, we dropped images with
    Background_without_leaves label, because these were not present in the original
    dataset.
    
    Original paper URL: https://arxiv.org/abs/1511.08060 Dataset URL:
    https://data.mendeley.com/datasets/tywbtsjrjv/1
    """,
    homepage='https://arxiv.org/abs/1511.08060',
    data_dir='C:\\Users\\ajlop\\tensorflow_datasets\\plant_village\\1.0.2',
    file_format=tfrecord,
    download_size=Unknown size,
    dataset_size=Unknown size,
    features=FeaturesDict({
        'image': Image(shape=(None, None, 3), d

## 3. Carga y preprocesamiento del dataset

Usamos **PlantVillage** desde `tensorflow_datasets`. El dataset completo tiene ~54.000 imágenes en
38 clases; para que el experimento sea **viable en Colab gratuito** nos centramos en el
subconjunto de **hojas de tomate** (problema multiclase coherente: tomate sano vs. varias
enfermedades) y submuestreamos a `MAX_PER_CLASS` imágenes por clase.

**Particiones:** dividimos de forma **estratificada** en Entrenamiento / Validación / Prueba
(70% / 15% / 15%), garantizando que cada clase esté representada en las tres particiones.

In [10]:
import sys

!{sys.executable} -m pip install importlib-resources

In [11]:
import sys

!{sys.executable} -m pip install kagglehub

In [12]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "emmarex/plantdisease"
)

print(dataset_path)

c:\Users\ajlop\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\ajlop\.cache\kagglehub\datasets\emmarex\plantdisease\versions\1


In [ ]:
# Descarga automática de PlantVillage (solo tiene split 'train') y lectura de metadatos.
ds_full, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True)

all_class_names = info.features["label"].names
print("Total de clases en PlantVillage:", len(all_class_names))

# Seleccionamos únicamente las clases de tomate (problema enfocado y manejable).
tomato_idx = [i for i, name in enumerate(all_class_names) if name.lower().startswith("tomato")]
tomato_idx_sorted = sorted(tomato_idx)
CLASS_NAMES = [all_class_names[i].replace("Tomato___", "").replace("_", " ")
               for i in tomato_idx_sorted]
NUM_CLASSES = len(CLASS_NAMES)

print(f"\nClases de tomate seleccionadas ({NUM_CLASSES}):")
for new_lbl, name in enumerate(CLASS_NAMES):
    print(f"  {new_lbl}: {name}")

META["num_clases"] = NUM_CLASSES
META["clases"] = CLASS_NAMES

In [14]:
from pathlib import Path

root = Path(dataset_path)

for item in root.iterdir():
    print(item)

C:\Users\ajlop\.cache\kagglehub\datasets\emmarex\plantdisease\versions\1\PlantVillage


In [15]:
from pathlib import Path

root = Path(dataset_path) / "PlantVillage"

print("Root:", root)

for item in root.iterdir():
    print(item.name)

Root: C:\Users\ajlop\.cache\kagglehub\datasets\emmarex\plantdisease\versions\1\PlantVillage
Pepper__bell___Bacterial_spot
Pepper__bell___healthy
PlantVillage
Potato___Early_blight
Potato___healthy
Potato___Late_blight
Tomato_Bacterial_spot
Tomato_Early_blight
Tomato_healthy
Tomato_Late_blight
Tomato_Leaf_Mold
Tomato_Septoria_leaf_spot
Tomato_Spider_mites_Two_spotted_spider_mite
Tomato__Target_Spot
Tomato__Tomato_mosaic_virus
Tomato__Tomato_YellowLeaf__Curl_Virus


In [17]:
from pathlib import Path
import os

#root = Path(dataset_path)

root = Path(dataset_path) / "PlantVillage"

all_images = []
all_labels = []

tomato_folders = []

tomato_folders = sorted([
    folder
    for folder in root.iterdir()
    if folder.is_dir() and folder.name.startswith("Tomato")
])


CLASS_NAMES = [f.name for f in tomato_folders]
NUM_CLASSES = len(CLASS_NAMES)

print("Clases encontradas:", NUM_CLASSES)

print(CLASS_NAMES)


for idx, folder in enumerate(tomato_folders):

    files = list(folder.glob("*.JPG"))
    files += list(folder.glob("*.jpg"))
    files += list(folder.glob("*.png"))

    files = files[:MAX_PER_CLASS]

    for img_path in files:

        img = tf.keras.utils.load_img(
            img_path,
            target_size=IMG_SIZE
        )

        img = tf.keras.utils.img_to_array(img)

        all_images.append(img.astype("uint8"))
        all_labels.append(idx)

X = np.array(all_images)
y = np.array(all_labels)

print("Forma X:", X.shape)
print("Forma y:", y.shape)

META["num_classes"] = NUM_CLASSES
META["classes"] = CLASS_NAMES



Clases encontradas: 10
['Tomato__Target_Spot', 'Tomato__Tomato_mosaic_virus', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_healthy', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite']
Forma X: (3000, 160, 160, 3)
Forma y: (3000,)


In [ ]:
# Mapeo de la etiqueta original de PlantVillage -> etiqueta local 0..NUM_CLASSES-1
orig_to_local = {orig: new for new, orig in enumerate(tomato_idx_sorted)}

# Recorremos el dataset una vez, redimensionamos a IMG_SIZE y llenamos "cubetas"
# por clase hasta MAX_PER_CLASS. Esto materializa un subconjunto pequeño y balanceado
# en memoria (apto para Colab), y nos permite hacer una partición estratificada con sklearn.
set_seeds(SEED)
buckets = defaultdict(list)
target_total = MAX_PER_CLASS * NUM_CLASSES

for image, label in tfds.as_numpy(ds_full):
    orig = int(label)
    if orig not in orig_to_local:
        continue
    local = orig_to_local[orig]
    if len(buckets[local]) >= MAX_PER_CLASS:
        continue
    img_resized = tf.image.resize(image, IMG_SIZE).numpy().astype("uint8")
    buckets[local].append(img_resized)
    if sum(len(v) for v in buckets.values()) >= target_total:
        break

# Construimos los arreglos X (imágenes) e y (etiquetas)
X = np.concatenate([np.stack(buckets[c]) for c in range(NUM_CLASSES)], axis=0)
y = np.concatenate([np.full(len(buckets[c]), c, dtype=np.int64) for c in range(NUM_CLASSES)])

print("Forma de X:", X.shape, "| Forma de y:", y.shape)
print("Imágenes por clase:")
for c in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[c]:<28} -> {len(buckets[c])}")

In [18]:
# Partición estratificada Train / Val / Test (70 / 15 / 15)
# Primero separamos el Test; luego dividimos el resto en Train y Val.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, stratify=y, random_state=SEED)

val_relative = VAL_SPLIT / (1.0 - TEST_SPLIT)  # ajustamos la proporción sobre el resto
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_relative, stratify=y_trainval, random_state=SEED)

print(f"Entrenamiento: {X_train.shape[0]} imágenes")
print(f"Validación:    {X_val.shape[0]} imágenes")
print(f"Prueba:        {X_test.shape[0]} imágenes")

META["particiones"] = {
    "train": int(X_train.shape[0]),
    "val": int(X_val.shape[0]),
    "test": int(X_test.shape[0]),
}

Entrenamiento: 2099 imágenes
Validación:    451 imágenes
Prueba:        450 imágenes


In [19]:
# Construcción de los DataLoaders con tf.data.
# Aplicamos preprocess_input de MobileNetV2 (lleva los píxeles al rango [-1, 1]).
AUTOTUNE = tf.data.AUTOTUNE


def make_dataset(X_arr, y_arr, training=False):
    """Crea un tf.data.Dataset con preprocesamiento, barajado (solo train), lotes y prefetch."""
    ds = tf.data.Dataset.from_tensor_slices((X_arr, y_arr))
    if training:
        ds = ds.shuffle(buffer_size=len(X_arr), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda img, lbl: (preprocess_input(tf.cast(img, tf.float32)), lbl),
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_val,   y_val,   training=False)
test_ds  = make_dataset(X_test,  y_test,  training=False)

# Capa de data augmentation (se aplica solo en entrenamiento, dentro del modelo).
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

print("DataLoaders listos.")

DataLoaders listos.


In [ ]:
# Visualización de ejemplos del conjunto de entrenamiento (antes de normalizar)
plt.figure(figsize=(12, 8))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i])
    plt.title(CLASS_NAMES[y_train[i]], fontsize=9)
    plt.axis("off")
plt.suptitle("Ejemplos del conjunto de entrenamiento", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print(CLASS_NAMES)

In [ ]:
for f in tomato_folders:
    print(f)

## 4. Arquitectura del modelo (MobileNetV2)

Definimos una función modular `build_model` que construye el mismo clasificador para ambas
configuraciones. El parámetro `fine_tune_at` controla la **estrategia de transferencia**:

- `fine_tune_at = None` → backbone **totalmente congelado** (Config A, extractor de características).
- `fine_tune_at = k` → se **descongelan** las capas del backbone a partir del índice `k` (Config B).

Nota técnica: invocamos el backbone con `training=False` para mantener las capas de
*BatchNormalization* en modo inferencia, práctica recomendada al hacer fine-tuning para no
desestabilizar las estadísticas preentrenadas.

In [ ]:
def build_model(num_classes: int, fine_tune_at=None):
    """Construye el modelo de transferencia sobre MobileNetV2.

    Parámetros
    ----------
    num_classes : int
        Número de clases de salida.
    fine_tune_at : int | None
        - None  -> backbone congelado (Config A: extractor de características).
        - int k -> se descongelan las capas del backbone desde el índice k (Config B).
    """
    base_model = MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,          # quitamos el clasificador original de ImageNet
        weights="imagenet",
    )

    if fine_tune_at is None:
        base_model.trainable = False                # Config A: todo congelado
    else:
        base_model.trainable = True                 # Config B: descongelamiento parcial
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False                 # capas inferiores quedan congeladas

    inputs = Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)                   # aumento de datos (solo en training)
    x = base_model(x, training=False)               # BatchNorm en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)                      # regularización para mitigar sobreajuste
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    return model, base_model


# Prueba rápida de construcción (Config A) y resumen de la arquitectura
_demo_model, _demo_base = build_model(NUM_CLASSES, fine_tune_at=None)
_demo_model.summary()
del _demo_model, _demo_base

## 5. Estrategia de entrenamiento y funciones de evaluación

- **Función de pérdida:** `sparse_categorical_crossentropy` (etiquetas enteras, salida softmax).
- **Optimizador:** Adam.
- **Métrica de monitoreo:** accuracy (validación), con *EarlyStopping* para evitar sobreajuste.
- **Registro de metadatos:** medimos el **tiempo de entrenamiento** y el número de épocas reales.

Las funciones `train_model`, `plot_history`, `evaluate_model` y `show_examples` son reutilizadas
por ambas configuraciones para garantizar una comparación justa.

In [ ]:
def compile_model(model, learning_rate):
    """Compila el modelo con Adam y entropía cruzada categórica dispersa."""
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


def train_model(model, epochs, initial_epoch=0, history_prev=None):
    """Entrena el modelo midiendo el tiempo y devuelve (history_dict, tiempo_segundos)."""
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True)

    start = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        initial_epoch=initial_epoch,
        callbacks=[early_stop],
        verbose=1,
    )
    elapsed = time.time() - start

    # Permite concatenar el historial de fases (head + fine-tuning) en Config B
    hist = history.history
    if history_prev is not None:
        hist = {k: history_prev.get(k, []) + v for k, v in hist.items()}
    return hist, elapsed


def count_trainable_params(model):
    """Devuelve (entrenables, totales) en número de parámetros."""
    trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    total = int(np.sum([np.prod(v.shape) for v in model.variables]))
    return trainable, total

In [ ]:
def plot_history(hist, title=""):
    """Grafica las curvas de aprendizaje (pérdida y accuracy) por época."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(hist["loss"], label="Entrenamiento")
    axes[0].plot(hist["val_loss"], label="Validación")
    axes[0].set_title(f"Pérdida (Loss) — {title}")
    axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(hist["accuracy"], label="Entrenamiento")
    axes[1].plot(hist["val_accuracy"], label="Validación")
    axes[1].set_title(f"Exactitud (Accuracy) — {title}")
    axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def get_predictions(model, ds):
    """Devuelve (y_true, y_pred) para un dataset dado."""
    y_true, y_pred = [], []
    for batch_x, batch_y in ds:
        probs = model.predict(batch_x, verbose=0)
        y_pred.extend(np.argmax(probs, axis=1))
        y_true.extend(batch_y.numpy())
    return np.array(y_true), np.array(y_pred)


def evaluate_model(model, ds, config_name=""):
    """Calcula accuracy, precision, recall y F1 (macro) e imprime el reporte de clasificación."""
    y_true, y_pred = get_predictions(model, ds)

    metrics = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1":        f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

    print(f"\n===== Métricas en PRUEBA — {config_name} =====")
    for k, v in metrics.items():
        print(f"  {k:<10}: {v:.4f}")
    print("\nReporte de clasificación por clase:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    return metrics, y_true, y_pred


def plot_confusion(y_true, y_pred, title=""):
    """Grafica la matriz de confusión."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Matriz de confusión — {title}")
    plt.xlabel("Predicción"); plt.ylabel("Real")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
def show_examples(model, correct=True, n=8, title=""):
    """Muestra ejemplos bien clasificados (correct=True) o mal clasificados (correct=False).

    Las imágenes se toman del conjunto de PRUEBA (X_test) en su forma original (uint8).
    """
    probs = model.predict(test_ds, verbose=0)
    preds = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)

    if correct:
        idxs = np.where(preds == y_test)[0]
    else:
        idxs = np.where(preds != y_test)[0]

    if len(idxs) == 0:
        print(f"No hay ejemplos {'correctos' if correct else 'incorrectos'} para mostrar.")
        return

    idxs = idxs[:n]
    plt.figure(figsize=(14, 7))
    for i, idx in enumerate(idxs):
        plt.subplot(2, 4, i + 1)
        plt.imshow(X_test[idx])
        color = "green" if correct else "red"
        plt.title(f"Real: {CLASS_NAMES[y_test[idx]]}\n"
                  f"Pred: {CLASS_NAMES[preds[idx]]} ({confidences[idx]:.2f})",
                  fontsize=9, color=color)
        plt.axis("off")
    estado = "BIEN clasificados" if correct else "MAL clasificados"
    plt.suptitle(f"Ejemplos {estado} — {title}", fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Configuración A — Extractor de características (backbone congelado)

El backbone MobileNetV2 permanece **completamente congelado**: solo se entrena el clasificador
final (`Dense`). Es la opción más rápida y de menor costo computacional, y sirve como **línea base
(referencia)** para interpretar si el fine-tuning aporta una mejora real.

In [ ]:
set_seeds(SEED)  # reproducibilidad: mismo punto de partida para ambas configuraciones

# Config A: backbone congelado (fine_tune_at=None)
model_A, base_A = build_model(NUM_CLASSES, fine_tune_at=None)
compile_model(model_A, learning_rate=LR_HEAD)

trainable_A, total_A = count_trainable_params(model_A)
print(f"Parámetros entrenables (Config A): {trainable_A:,} de {total_A:,}")

history_A, time_A = train_model(model_A, epochs=EPOCHS_HEAD)
print(f"\nTiempo de entrenamiento Config A: {time_A:.1f} s")

In [ ]:
plot_history(history_A, title="Config A (congelado)")

## 7. Configuración B — Fine-tuning parcial (descongelamiento)

Se entrena en **dos fases**, siguiendo la buena práctica vista en clase:

1. **Fase 1 (calentamiento):** con el backbone congelado se entrena el head (igual que Config A).
   Esto evita que gradientes grandes y aleatorios del clasificador recién inicializado dañen los
   pesos preentrenados.
2. **Fase 2 (fine-tuning):** se **descongelan** las capas superiores del backbone (a partir de
   `FINE_TUNE_AT`) y se reentrena con un *learning rate* muy bajo (`1e-5`) para ajustar
   finamente las características al dominio de hojas de tomate.

In [ ]:
set_seeds(SEED)  # mismo punto de partida que Config A

# ---- Fase 1: calentamiento del head con el backbone congelado ----
model_B, base_B = build_model(NUM_CLASSES, fine_tune_at=None)
compile_model(model_B, learning_rate=LR_HEAD)

trainable_B1, _ = count_trainable_params(model_B)
print(f"[Fase 1] Parámetros entrenables: {trainable_B1:,}")
history_B, time_B1 = train_model(model_B, epochs=EPOCHS_HEAD)

In [ ]:
# ---- Fase 2: descongelamiento parcial y fine-tuning con learning rate bajo ----
base_B.trainable = True
for layer in base_B.layers[:FINE_TUNE_AT]:
    layer.trainable = False     # mantenemos congeladas las capas inferiores (rasgos genéricos)

# Recompilamos para que el cambio de 'trainable' surta efecto, con LR bajo
compile_model(model_B, learning_rate=LR_FINETUNE)

trainable_B2, total_B = count_trainable_params(model_B)
print(f"[Fase 2] Parámetros entrenables: {trainable_B2:,} de {total_B:,}")
print(f"Descongelando desde la capa {FINE_TUNE_AT} de {len(base_B.layers)} del backbone.")

# Continuamos el entrenamiento concatenando el historial de la fase 1
total_epochs = EPOCHS_HEAD + EPOCHS_FT
history_B, time_B2 = train_model(
    model_B, epochs=total_epochs, initial_epoch=len(history_B["loss"]),
    history_prev=history_B)

time_B = time_B1 + time_B2
print(f"\nTiempo total de entrenamiento Config B: {time_B:.1f} s "
      f"(fase 1: {time_B1:.1f}s + fase 2: {time_B2:.1f}s)")

In [ ]:
# La línea vertical marca el inicio de la fase de fine-tuning (descongelamiento)
plot_history(history_B, title="Config B (fine-tuning parcial)")
print(f"(El fine-tuning comenzó en la época {EPOCHS_HEAD})")

## 8. Evaluación comparativa en el conjunto de PRUEBA

Evaluamos ambas configuraciones sobre el **mismo conjunto de prueba** (nunca visto durante el
entrenamiento) y comparamos: Accuracy, Precision, Recall, F1-score (macro), matrices de confusión,
y ejemplos bien y mal clasificados.

In [ ]:
# Métricas en prueba para cada configuración
metrics_A, y_true_A, y_pred_A = evaluate_model(model_A, test_ds, "Config A (congelado)")
metrics_B, y_true_B, y_pred_B = evaluate_model(model_B, test_ds, "Config B (fine-tuning)")

In [ ]:
# Matrices de confusión de ambas configuraciones
plot_confusion(y_true_A, y_pred_A, "Config A (congelado)")
plot_confusion(y_true_B, y_pred_B, "Config B (fine-tuning)")

In [ ]:
# Ejemplos bien y mal clasificados (requisito de visión por computador).
# Usamos el mejor modelo (Config B) para el análisis cualitativo de errores.
show_examples(model_B, correct=True,  n=8, title="Config B")
show_examples(model_B, correct=False, n=8, title="Config B")

In [ ]:
# Tabla resumen: desempeño vs. costo computacional (juicio de ingeniería)
resumen = pd.DataFrame([
    {
        "Configuración": "A — Congelado",
        "Accuracy":  round(metrics_A["accuracy"], 4),
        "Precision": round(metrics_A["precision"], 4),
        "Recall":    round(metrics_A["recall"], 4),
        "F1 (macro)": round(metrics_A["f1"], 4),
        "Tiempo (s)": round(time_A, 1),
        "Params entrenables": trainable_A,
    },
    {
        "Configuración": "B — Fine-tuning",
        "Accuracy":  round(metrics_B["accuracy"], 4),
        "Precision": round(metrics_B["precision"], 4),
        "Recall":    round(metrics_B["recall"], 4),
        "F1 (macro)": round(metrics_B["f1"], 4),
        "Tiempo (s)": round(time_B, 1),
        "Params entrenables": trainable_B2,
    },
])

# Guardamos los metadatos finales del experimento
META.update({
    "epocas_head": EPOCHS_HEAD,
    "epocas_finetune": EPOCHS_FT,
    "optimizador": "Adam",
    "lr_head": LR_HEAD,
    "lr_finetune": LR_FINETUNE,
    "tecnica_transferencia": "A: feature extraction | B: fine-tuning parcial",
    "tiempo_entrenamiento_A_s": round(time_A, 1),
    "tiempo_entrenamiento_B_s": round(time_B, 1),
})

print("===== RESUMEN COMPARATIVO =====")
display(resumen)
print("\n===== METADATOS DEL EXPERIMENTO =====")
for k, v in META.items():
    print(f"  {k}: {v}")

In [ ]:
# Gráfico de barras: F1-score vs. tiempo de entrenamiento (relación desempeño / costo)
fig, ax1 = plt.subplots(figsize=(8, 5))
configs = resumen["Configuración"]
x = np.arange(len(configs))

bars = ax1.bar(x - 0.2, resumen["F1 (macro)"], width=0.4, label="F1 (macro)", color="#4C72B0")
ax1.set_ylabel("F1-score (macro)", color="#4C72B0")
ax1.set_ylim(0, 1)
for b, v in zip(bars, resumen["F1 (macro)"]):
    ax1.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

ax2 = ax1.twinx()
bars2 = ax2.bar(x + 0.2, resumen["Tiempo (s)"], width=0.4, label="Tiempo (s)", color="#DD8452")
ax2.set_ylabel("Tiempo de entrenamiento (s)", color="#DD8452")

ax1.set_xticks(x); ax1.set_xticklabels(configs)
plt.title("Desempeño (F1) vs. Costo computacional (tiempo)")
fig.tight_layout()
plt.show()

## 9. Análisis e interpretación de resultados (PASO 3)

> _Rellena los espacios `[...]` con los valores obtenidos en tu ejecución. Las preguntas guía te
> ayudan a construir el juicio de ingeniería que pide el profesor._

### 9.1 Explicación del rendimiento (¿overfitting? ¿aprovechó el Transfer Learning?)

**Cómo leer las curvas de aprendizaje:**
- **Buen Transfer Learning:** la accuracy de validación sube rápido en las primeras épocas y se
  estabiliza alta; la pérdida de validación baja de forma sostenida. Indica que las
  características de ImageNet eran útiles para hojas de tomate.
- **Sobreajuste (overfitting):** la pérdida de *entrenamiento* sigue bajando mientras la de
  *validación* se estanca o **sube**, y se abre una brecha (gap) entre ambas curvas. En Config B
  vigila especialmente la fase de fine-tuning (a partir de la época `EPOCHS_HEAD`), donde el
  riesgo aumenta al haber más parámetros entrenables.
- **Subajuste (underfitting):** ambas curvas se quedan en valores bajos → el modelo no tiene
  capacidad o no entrenó suficiente.

**Plantilla de redacción:**
> La Configuración A alcanzó un F1 de `[...]` y la Configuración B de `[...]`. La curva de
> validación de A se estabilizó alrededor de la época `[...]`, sin brecha notable respecto a
> entrenamiento, lo que sugiere `[buen ajuste / subajuste]`. En B, tras el descongelamiento en la
> época `[EPOCHS_HEAD]`, la métrica de validación `[mejoró / se estancó]`, y la brecha
> train–val fue de `[...]`, indicando `[sobreajuste leve / ausencia de sobreajuste]`. Esto sugiere
> que el modelo `[sí / no]` aprovechó el aprendizaje por transferencia, porque `[...]`.

### 9.2 Análisis de la matriz de confusión y errores
- ¿Qué clases se confunden más entre sí? Frecuentemente enfermedades visualmente similares
  (p. ej. distintos tipos de *spot*/*blight*). Anota los pares: `[clase X ↔ clase Y]`.
- Revisa los **ejemplos mal clasificados**: ¿son imágenes ambiguas, con mala iluminación, o
  síntomas en etapa temprana? Esto justifica los errores ante el profesor.

## 10. Juicio de ingeniería y recomendaciones (PASO 3.2 / Criterio 4)

### 10.1 Relación desempeño vs. costo computacional
Contrasta la mejora de desempeño con el costo adicional:

| Aspecto | Config A (congelado) | Config B (fine-tuning) |
|---|---|---|
| F1-score (macro) | `[...]` | `[...]` |
| Tiempo de entrenamiento | `[...] s` | `[...] s` |
| Parámetros entrenables | `[...]` | `[...]` |
| Riesgo de sobreajuste | Bajo | Mayor |
| Complejidad de implementación | Menor | Mayor (2 fases) |

**Pregunta clave:** ¿la mejora de F1 (`[ΔF1 = ...]`) *justifica* el incremento de tiempo
(`[Δt = ...]`)? 

### 10.2 Recomendación basada en evidencia (plantilla)
> A partir de la evidencia experimental, recomendaríamos adoptar la **Configuración `[A / B]`**
> porque `[obtuvo mejor F1 / ofreció una relación desempeño-costo más favorable]`. La
> Configuración B logró una mejora de `[...]` puntos de F1 a un costo de `[...]`× más tiempo de
> entrenamiento, lo cual es `[razonable / excesivo]` en un escenario de `[despliegue en campo con
> recursos limitados / servidor con GPU]`.
>
> **Condiciones de uso:** el modelo es adecuado cuando `[...]`. **Limitaciones:** `[dataset de
> laboratorio con fondo uniforme; puede degradarse con fotos reales de campo; clases
> desbalanceadas; etc.]`. **Mejoras futuras:** `[probar más data augmentation, EfficientNet,
> descongelar más capas, recolectar imágenes de campo, validación cruzada]`.

> Recuerda (Criterio 4 del profesor): **las recomendaciones deben derivarse del experimento**; una
> recomendación genérica no se evalúa positivamente.

In [ ]:
# (Opcional) Guardar resultados y metadatos para el informe escrito
import json

resumen.to_csv("resultados_comparativos.csv", index=False)
with open("metadatos_experimento.json", "w", encoding="utf-8") as f:
    json.dump({k: (str(v) if not isinstance(v, (int, float, list, dict)) else v)
               for k, v in META.items()}, f, indent=2, ensure_ascii=False)

print("Guardado: resultados_comparativos.csv y metadatos_experimento.json")
print("\n¡Experimento completo! Usa estos archivos y las gráficas para llenar el informe.")